In [18]:
from typing import Annotated, TypedDict
from langgraph.graph import START, END, StateGraph
from langchain_groq import ChatGroq
import operator

In [19]:
llm = ChatGroq(model="groq-m3o", temperature=0.4)

In [20]:
class State(TypedDict):
    n: int
    history: Annotated[list[int], operator.add]

In [21]:
def double(state: State) -> dict:
    doubled_value = state['n'] * 2
    # n is updates to the doubled value and history is updated with the new value
    return {
        "n": doubled_value,
        "history": [doubled_value]
    }

In [22]:
# routing function
def	should_continue(state:	State)	->	str: 
    if state["n"] > 100:
        return	END 
    return	"double"

In [23]:
builder = StateGraph(State)
builder.add_node("double", double)

builder.add_edge(START, "double")
builder.add_conditional_edges("double", 
                              should_continue, 
                              {
                                "double": "double",
                                END: END
                              })



In [24]:
graph = builder.compile()


In [25]:
final = graph.invoke({
    "n": 2, 
    "history": []
    })

print(final['history'])

[4, 8, 16, 32, 64, 128]
